In [ ]:
# Las tablas puente permiten preguntar por autores sin duplicar documentos.

from pathlib import Path
import json
import sqlite3

import pandas as pd

ACTIVITY_DIR = next(candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "data").is_dir() and (candidate / "submission").is_dir())
DATABASE_FILE = ACTIVITY_DIR / "data" / "scopus_proptech.db"
SUBMISSION_DIR = ACTIVITY_DIR / "submission"

In [ ]:
# El diagrama hace explícito el camino que seguirá cada JOIN.

# authors <- document_authors -> documents -> sources

database = sqlite3.connect(DATABASE_FILE)

In [ ]:
# El primer JOIN conecta cada autor con los documentos que publicó.

authors_by_documents = pd.read_sql_query(
    """
    SELECT a.author_name, COUNT(DISTINCT da.document_id) AS document_count
    FROM authors AS a
    JOIN document_authors AS da ON a.author_id = da.author_id
    GROUP BY a.author_id, a.author_name
    ORDER BY document_count DESC, a.author_name
    LIMIT 20
    """,
    database,
)
authors_by_documents

In [ ]:
# El segundo JOIN conecta documentos con la fuente donde aparecen.

sources_by_documents = pd.read_sql_query(
    """
    SELECT s.source_title, COUNT(DISTINCT d.document_id) AS document_count
    FROM sources AS s
    JOIN documents AS d ON s.source_id = d.source_id
    GROUP BY s.source_id, s.source_title
    ORDER BY document_count DESC, s.source_title
    LIMIT 20
    """,
    database,
)
sources_by_documents

In [ ]:
# Las respuestas persistentes permiten revisar la consulta después de la clase.

authors_by_documents.to_csv(SUBMISSION_DIR / "authors_by_documents.csv", index=False)
sources_by_documents.to_csv(SUBMISSION_DIR / "sources_by_documents.csv", index=False)

questions = [
    {"pregunta": "¿Qué autores concentran más documentos sobre proptech?", "archivo_respuesta": "authors_by_documents.csv"},
    {"pregunta": "¿Qué fuentes concentran más documentos sobre proptech?", "archivo_respuesta": "sources_by_documents.csv"},
]
(SUBMISSION_DIR / "questions.json").write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8")
database.close()